# Analyse incrémentale exhaustive des familles de facteurs — STOXX EUROPE 600

Ce notebook teste chaque variable de `FACTOR_FAMILIES` avec le facteur historique de sa propre famille.

Pour chaque famille et chaque dimension, chaque variable est ajoutée séparément avec une pondération **1:1** :
- facteur historique de la famille : `level=1.0` ;
- variable candidate : une seule dimension active à `1.0`.

Les dimensions testées sont le niveau, puis tous les changements `pct`, `diff` et `rank_diff` sur les horizons 1, 3, 6 et 12 mois. Les résultats sont imprimés sous forme de tables compactes ; aucune figure n'est produite.


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from factor_config import FACTOR_FAMILIES, LOWER_IS_BETTER, signal_options
from func import (
    calculate_benchmark_performance,
    export_backtest_results,
    load_backtest_data,
    test_incremental_signals,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "func.py").exists():
    raise RuntimeError(
        "Définissez le répertoire de travail Jupyter sur C:\\dev\\factor_backtest."
    )

MARKET = "STOXX EUROPE 600"
BENCHMARK = "STOXX EUROPE 600"
START_DATE = "2007-12-01"
PERCENTILE = 0.13
N_JOBS = 8
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
OUTPUT_NAME = "factor_families_incremental_STOXX600"
EXPORT_ROOT = REPO_ROOT / "exports"
LIST_NOIRE_PATH = None

CANDIDATE_DIMENSIONS = (
    "level",
    "pct_1", "pct_3", "pct_6", "pct_12",
    "diff_1", "diff_3", "diff_6", "diff_12",
    "rank_diff_1", "rank_diff_3", "rank_diff_6", "rank_diff_12",
)

BASELINE_CANDIDATES = {
    "growth": ("GROWTH_SCORE_FS_SECTOR", "Growth Avg Percentile"),
    "quality": ("Quality Avg Percentile", "MARGIN_SCORE_FS_SECTOR"),
    "value": ("VALUE_SCORE_FS_SECTOR", "Value Avg Percentile"),
    "momentum": ("MOMENTUM_SCORE_FS_SECTOR", "Mom Avg Percentile"),
    "lowvol": ("LOW_VOL_SCORE_FS_SECTOR", "LowVol Avg Percentile"),
    "dividend": ("Dividend Avg Percentile", "Dividend_NTM Avg Percentile"),
    "size": ("Size Avg Percentile",),
}

KEY_METRICS = (
    "active_cagr",
    "top_information_ratio",
    "top_worst_cagr",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "top_annualized_volatility",
    "top_max_drawdown",
)

available_columns = set(pq.ParquetFile(REPO_ROOT / "data" / "screen_aggregate.parquet").schema_arrow.names)
BASELINE_COLUMNS = {}
for family, candidates in BASELINE_CANDIDATES.items():
    selected = next((candidate for candidate in candidates if candidate in available_columns), None)
    if selected is None:
        raise KeyError(f"Aucun facteur historique disponible pour {family}: {candidates}")
    BASELINE_COLUMNS[family] = selected

missing_by_family = {
    family: [variable for variable in variables if variable not in available_columns]
    for family, variables in FACTOR_FAMILIES.items()
}
missing_by_family = {
    family: missing for family, missing in missing_by_family.items() if missing
}
if missing_by_family:
    raise KeyError(f"Variables FACTOR_FAMILIES absentes du screen: {missing_by_family}")

def _safe_token(value):
    """Construit un identifiant portable pour les colonnes candidates."""
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_") or "variable"

def _candidate_alias(family, dimension, variable):
    """Crée un alias distinct pour tester aussi les changements du facteur historique."""
    return f"__incremental__{family}__{dimension}__{_safe_token(variable)}"

def _baseline_config(family):
    """Construit le facteur historique de la famille avec un poids égal à un."""
    return {
        BASELINE_COLUMNS[family]: signal_options(
            level=1.0,
            higher_is_better=True,
        )
    }

print(f"Marché: {MARKET} | Benchmark: {BENCHMARK}")
print(f"Familles: {list(FACTOR_FAMILIES)}")
print(f"Dimensions candidates: {len(CANDIDATE_DIMENSIONS)}")
print(f"Baselines résolues: {BASELINE_COLUMNS}")


In [ ]:
DATA_DIR = REPO_ROOT / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"

raw_variables = list(dict.fromkeys(
    variable
    for variables in FACTOR_FAMILIES.values()
    for variable in variables
))
load_variables = list(dict.fromkeys(raw_variables + list(BASELINE_COLUMNS.values())))

screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=load_variables,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

missing_after_load = [
    variable for variable in load_variables if variable not in screen.columns
]
if missing_after_load:
    raise KeyError(f"Variables absentes après chargement: {missing_after_load}")
if f"Weight in {BENCHMARK}" not in screen.columns:
    raise KeyError(f"La colonne Weight in {BENCHMARK} est absente du screen.")

benchmark_performance = calculate_benchmark_performance(
    screen=screen,
    returns=returns,
    bench=BENCHMARK,
    start_date=START_DATE,
)

RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": benchmark_performance,
    "percentile": PERCENTILE,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": {},
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": False,
    "build_figure": False,
}

incremental_batches = {"by_family": {}}
candidate_manifest_rows = []

for family, variables in FACTOR_FAMILIES.items():
    incremental_batches["by_family"][family] = {}
    for dimension in CANDIDATE_DIMENSIONS:
        candidate_config = {}
        for variable in variables:
            alias = _candidate_alias(family, dimension, variable)
            screen[alias] = screen[variable]
            candidate_config[alias] = signal_options(
                higher_is_better=variable not in LOWER_IS_BETTER,
                **{dimension: 1.0},
            )
            candidate_manifest_rows.append({
                "market": MARKET,
                "family": family,
                "baseline_variable": BASELINE_COLUMNS[family],
                "variable": variable,
                "dimension": dimension,
                "alias": alias,
                "higher_is_better": variable not in LOWER_IS_BETTER,
                "weight_baseline": 1.0,
                "weight_candidate": 1.0,
            })

        incremental_batches["by_family"][family][dimension] = test_incremental_signals(
            screen=screen,
            returns=returns,
            baseline_config=_baseline_config(family),
            candidate_config=candidate_config,
            list_noire_path=LIST_NOIRE_PATH,
            **RUN_OPTIONS,
        )
        screen = incremental_batches["by_family"][family][dimension]["screen"]

candidate_manifest = pd.DataFrame(candidate_manifest_rows)
print(f"Observations screen: {screen.shape} | returns: {returns.shape}")
print(f"Lots incrémentaux: {len(FACTOR_FAMILIES) * len(CANDIDATE_DIMENSIONS)}")
print(f"Candidats variables × dimensions: {len(candidate_manifest)}")


In [ ]:
all_results = {"incremental": incremental_batches}
exported = export_backtest_results(
    results=all_results,
    output_dir=EXPORT_ROOT,
    export_name=OUTPUT_NAME,
    export_html=False,
    export_png=False,
    export_holdings=False,
)
EXPORT_DIR = Path(exported["export_dir"])
metrics = exported["metrics"].copy()

candidate_lookup = candidate_manifest.set_index("alias").to_dict(orient="index")
metric_columns = [column for column in KEY_METRICS if column in metrics.columns]
incremental_rows = []

for test_group, group_rows in metrics.groupby("test_group", dropna=True):
    group_text = str(test_group)
    group_parts = group_text.split(" / ")
    if len(group_parts) < 3 or group_parts[0] != "incremental":
        continue
    family, dimension = group_parts[-2], group_parts[-1]
    if family not in FACTOR_FAMILIES or dimension not in CANDIDATE_DIMENSIONS:
        continue

    baseline_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_baseline")
    ]
    candidate_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_candidate")
    ]
    for _, candidate in candidate_rows.iterrows():
        lookup = candidate_lookup.get(str(candidate["test_name"]))
        if lookup is None:
            raise KeyError(
                f"Alias candidat absent du manifest: {candidate['test_name']}"
            )
        baseline = baseline_rows.loc[
            (baseline_rows["period_id"].eq(candidate["period_id"]))
            & (baseline_rows["scope"].eq(candidate["scope"]))
        ]
        if baseline.empty:
            continue
        baseline = baseline.iloc[0]
        row = {
            **lookup,
            "period_id": candidate["period_id"],
            "scope": candidate["scope"],
            "period_label": candidate.get("period_label"),
            "candidate_test_path": candidate["test_path"],
            "baseline_test_path": baseline["test_path"],
        }
        for column in metric_columns:
            candidate_value = pd.to_numeric(candidate.get(column), errors="coerce")
            baseline_value = pd.to_numeric(baseline.get(column), errors="coerce")
            row[f"{column}_candidate"] = candidate_value
            row[f"{column}_baseline"] = baseline_value
            row[f"delta_{column}"] = candidate_value - baseline_value

        row["incremental_perf_improved"] = (
            row["delta_active_cagr"] > 0
            and row["delta_top_information_ratio"] > 0
            and row["delta_top_worst_cagr"] > 0
        )
        row["incremental_risk_not_worse"] = (
            row["delta_active_max_drawdown"] <= 0
            and row["delta_tracking_error_annualized"] <= 0
        )
        row["incremental_absolute_risk_not_worse"] = (
            row.get("delta_top_annualized_volatility", np.nan) <= 0
            and row.get("delta_top_max_drawdown", np.nan) <= 0
        )
        incremental_rows.append(row)

incremental_effects = pd.DataFrame(incremental_rows)
if incremental_effects.empty:
    raise RuntimeError("Aucune ligne incrémentale n'a été produite.")

incremental_effects_all_periods = incremental_effects.sort_values(
    ["family", "variable", "dimension", "scope", "period_id"]
).reset_index(drop=True)
incremental_effects_total = incremental_effects.loc[
    incremental_effects["period_id"].eq("total")
].sort_values(
    ["family", "delta_active_cagr"],
    ascending=[True, False],
).reset_index(drop=True)

subperiods = incremental_effects.loc[
    incremental_effects["scope"].eq("subperiod")
].copy()
if not subperiods.empty:
    consistency = subperiods.groupby(
        ["family", "baseline_variable", "variable", "dimension", "higher_is_better"],
        dropna=False,
    ).agg(
        subperiod_count=("period_id", "count"),
        positive_active_cagr_rate=("delta_active_cagr", lambda values: float((values > 0).mean())),
        positive_information_ratio_rate=("delta_top_information_ratio", lambda values: float((values > 0).mean())),
        positive_worst_cagr_rate=("delta_top_worst_cagr", lambda values: float((values > 0).mean())),
        performance_improved_rate=("incremental_perf_improved", "mean"),
        risk_not_worse_rate=("incremental_risk_not_worse", "mean"),
        absolute_risk_not_worse_rate=("incremental_absolute_risk_not_worse", "mean"),
    ).reset_index()
    strict_mask = (
        subperiods["incremental_perf_improved"]
        & subperiods["incremental_risk_not_worse"]
        & subperiods["incremental_absolute_risk_not_worse"]
    )
    strict_rates = subperiods.assign(strict_improvement=strict_mask).groupby(
        ["family", "baseline_variable", "variable", "dimension"],
        dropna=False,
    )["strict_improvement"].mean().rename("strict_improvement_rate")
    consistency = consistency.merge(
        strict_rates.reset_index(),
        on=["family", "baseline_variable", "variable", "dimension"],
        how="left",
    )
else:
    consistency = pd.DataFrame()

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
candidate_manifest.to_csv(EXPORT_DIR / "candidate_manifest.csv", index=False)
incremental_effects_all_periods.to_csv(
    EXPORT_DIR / "incremental_effects_all_periods.csv",
    index=False,
)
incremental_effects_total.to_csv(
    EXPORT_DIR / "incremental_effects_total.csv",
    index=False,
)
consistency.to_csv(EXPORT_DIR / "incremental_consistency.csv", index=False)

run_manifest = {
    "market": MARKET,
    "benchmark": BENCHMARK,
    "percentile": PERCENTILE,
    "n_jobs": N_JOBS,
    "weighting": "baseline level 1.0 + one candidate dimension 1.0",
    "candidate_dimensions": list(CANDIDATE_DIMENSIONS),
    "families": list(FACTOR_FAMILIES),
    "baseline_columns": BASELINE_COLUMNS,
    "outputs": [
        "backtest_metrics.csv",
        "backtest_registry.json",
        "candidate_manifest.csv",
        "incremental_effects_all_periods.csv",
        "incremental_effects_total.csv",
        "incremental_consistency.csv",
    ],
}
with (EXPORT_DIR / "run_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)

total_display_columns = [
    "family",
    "baseline_variable",
    "variable",
    "dimension",
    "delta_active_cagr",
    "delta_top_information_ratio",
    "delta_top_worst_cagr",
    "delta_robust_score",
    "delta_active_max_drawdown",
    "delta_tracking_error_annualized",
    "incremental_perf_improved",
    "incremental_risk_not_worse",
    "incremental_absolute_risk_not_worse",
]
consistency_display_columns = [
    "family",
    "baseline_variable",
    "variable",
    "dimension",
    "subperiod_count",
    "positive_active_cagr_rate",
    "positive_information_ratio_rate",
    "positive_worst_cagr_rate",
    "performance_improved_rate",
    "risk_not_worse_rate",
    "absolute_risk_not_worse_rate",
    "strict_improvement_rate",
]

print(f"Répertoire des résultats: {EXPORT_DIR}")
print("Tableau total: chaque ligne est une variable × dimension.")
display(incremental_effects_total.loc[
    :, [column for column in total_display_columns if column in incremental_effects_total.columns]
])
print("Tableau de persistance: taux de périodes positives.")
display(consistency.loc[
    :, [column for column in consistency_display_columns if column in consistency.columns]
])
